### **Prueba de Conocimiento**

---


**Cargo:** Analista de Riesgo de Crédito

**Nombre:** Maria Paula Guerrero Suárez


### **Antes de ejecutar:**


Se deben crear las siguientes carpetas en Descargas:

- archivos_zips: se deben colocar los .zip de los Indicadores gerenciales - NIIF descargados de la SFC (dic-2021, dic-2022, dic-2023, dic-2024, sep-2024 y sep-2025).

- archivos_excel: será el destino donde se extraen los excels que se encuentran comprimidos inicialmente.

- bases_consolidadas: será el destino donde quedará el consolidado final.

Se deben instalar las siguientes dependencias si no se encuentran:

- pip install xlrd openpyxl
- pip install XlsxWriter

In [47]:
#Librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import os

In [48]:
#Descomprimir archivos

ruta_zips = r"C:\Users\DELL\Downloads\archivos_zips"  # Se debe cambiar por ruta de descarga que se vaya a utilizar.
ruta_excel = r"C:\Users\DELL\Downloads\archivos_excel" # Se debe cambiar por ruta de carpeta creada "archivos_excel" (Se almacenarán los archivos descomprimidos)

os.makedirs(ruta_excel, exist_ok=True)

for archivo in os.listdir(ruta_zips):
    if archivo.endswith(".zip"):
        with zipfile.ZipFile(os.path.join(ruta_zips, archivo), 'r') as zip_ref:
            zip_ref.extractall(ruta_excel)

print("Archivos descomprimidos correctamente")

Archivos descomprimidos correctamente


Los indicadores gerenciales de la Superintendencia Financiera se descargan en formato comprimido (.zip).

Este código automatiza la extracción de los archivos Excel contenidos en cada descarga.

Para esto es necesario descargar los archivos de indicadores gerenciales .zip correspondientes a las siguientes fechas (dic-2021, dic-2022, dic-2023, dic-2024, sep-2024 y sep-2025).

In [49]:
#Función de limpiar los archivos

def limpiar_archivo(ruta_archivo):

    df = pd.read_excel(ruta_archivo, header=None, engine="xlrd")

    # Identificar fila donde empieza el balance - información
    inicio = df[df.iloc[:, 0].astype(str).str.contains("BALANCE", na=False)].index[0]

    # Fila donde aparecen los nombres de las entidades
    fila_entidades = inicio - 2

    entidades = df.iloc[fila_entidades, 3:].tolist()
    
    #Tomar únicamente los datos sin encabezados

    data = df.iloc[inicio + 1:].copy()

    data = data[[0,1,2] + list(range(3, len(entidades)+3))]

    data.columns = ["rubro","subrubro","detalle"] + entidades

    # Eliminar filas completamente vacías
    data = data.dropna(how="all")

    #Completar los niveles

    data["Rubro"] = data["rubro"].ffill()
    data["Subrubro"] = data["subrubro"].ffill()
    data["Detalle"] = data["detalle"]

    #Dataframe con entidades y valor en columnas, se encontraba en formato horizontal.

    df_limpio = data.melt(
        id_vars = ["Rubro", "Subrubro", "Detalle"],
        value_vars = entidades,
        var_name = "Entidad",
        value_name = "Valor"
)

    return df_limpio

Se ordenan los datos para colocarlos en estructura de Rubro, Subrubro y Detalle.

Además, se transforman las entidades de columnas a filas para poder realizar el análisis, en las bases originales se encontraban las entidades en horizontal, es decir, en columnas, para hacer un análisis completo, se transformó la base de datos obteniendo una columna con el nombre de emisor y otra con su valor. 

In [50]:
#Consolidación de archivos seleccionados (periodos)

lista_periodos_limpios = []

for archivo in os.listdir(ruta_excel):

    if archivo.endswith(".xls"):

        ruta = os.path.join(ruta_excel, archivo)

        df_periodo = limpiar_archivo(ruta)

        # Extraer fecha desde nombre del archivo ya que todos los descomprimidas la contienen.
        nombre = archivo.replace(".xls","")
        anio, mes = nombre.split("_")[-2:]

        fecha = pd.Timestamp(int(anio), int(mes), 1) + pd.offsets.MonthEnd(0)

        df_periodo["Fecha"] = fecha

        lista_periodos_limpios.append(df_periodo)

df_consolidado = pd.concat(lista_periodos_limpios, ignore_index=True)

La fecha asignada corresponde al cierre del período reportado por la Superintendencia Financiera, ya que los indicadores representan el estado financiero al final del mes.

Se automatiza la lectura de todos los archivos descargados y se consolida la información en una única base de datos limpia, permitiendo analizar la evolución de los indicadores financieros a través del tiempo.

In [51]:
df_consolidado["Fecha"] = pd.to_datetime(df_consolidado["Fecha"])

df_consolidado = df_consolidado.sort_values(["Fecha","Entidad"]).reset_index(drop=True)

Se ordena la base de datos por fecha y entidad para facilitar el análisis.

In [52]:
# Se crea una copia para evitar modificar la base original durante el proceso de exportación
df_exportar = df_consolidado.copy()
df_exportar.insert(0, "Index", range(1, len(df_exportar)+1))
df_exportar = df_exportar[["Index", "Fecha", "Rubro", "Subrubro", "Detalle", "Entidad", "Valor"]]

Se da formato de fecha dd/mm/aaaa, y se incorpora un index y se ordenan las columnas para facilitar la lectura siguiendo la estructura propuesta de la imagen.

Ordenar las columnas.

In [53]:
# Se reemplaza NaN por vacío ("") para que en el Excel quede con las celdas faltantes en blanco
df_exportar = df_exportar.replace({np.nan: ""})

In [54]:
df_consolidado["Valor"].isna().sum()

14389

In [56]:
# Obtener rango de periodos de la base consolidada
periodo_inicio = df_consolidado["Fecha"].min().strftime("%b%Y")
periodo_fin = df_consolidado["Fecha"].max().strftime("%b%Y")

# Crear nombre del archivo
nombre_archivo = f"base_final_{periodo_inicio}_{periodo_fin}.xlsx"

#Ruta bases consolidadas final
carpeta_bases_final = r"C:\Users\DELL\Downloads\bases_consolidadas" #Se debe crear la carpeta "bases_consolidadas"

os.makedirs(carpeta_bases_final, exist_ok=True)

ruta_salida = os.path.join(carpeta_bases_final, nombre_archivo)

#Se desea obtener la columna de valores en formato Moneda, por lo tanto se aplica la siguiente lógica

with pd.ExcelWriter(ruta_salida, engine="xlsxwriter") as writer:
        
    df_exportar.to_excel(writer, index=False, sheet_name="Base_Indicadores")

    workbook = writer.book
    worksheet = writer.sheets["Base_Indicadores"]

    formato_moneda = workbook.add_format({'num_format': '$#,##0.00'})

    # Aplicar formato a la columna G (Valor)
    worksheet.set_column('G:G', None, formato_moneda)
    

print("Archivo exportado:", ruta_salida)
print("Filas:", len(df_exportar), "| Columnas:", df_exportar.shape[1])

Archivo exportado: C:\Users\DELL\Downloads\bases_consolidadas\base_final_Dec2021_Sep2025.xlsx
Filas: 271269 | Columnas: 7


Se crea una carpeta de salida llamada "bases_consolidadas" para así, tener las bases consolidadas con sus fechas de inicio a fin.

Se obtiene el archivo, se descarga para proceder con análisis.

Se puede observar como en el excel exportado se encuentran algunos valores en blanco, sin embargo, se decide mantenerlos como vacíos para preservar la estructura de los indicadores y permitir comparabilidad entre entidades financieras.